## Q5 - 10x10 Cleaning Robot (Q-Learning vs SARSA)


Environment Definition (Grid, Obstacles, Dirt, Rewards)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Part (a): Grid size
H, W = 10, 10
start_r, start_c = 0, 0

# Actions: 0=Up, 1=Down, 2=Left, 3=Right, 4=Clean
A = 5

# Discount factor
gamma = 0.95

# Redirty after this many steps (only for initially dirty cells)
REDIRTY_STEPS = 20

# Rewards (simple and reasonable)
R_STEP = -1        # normal move cost
R_INVALID = -3     # trying to move into wall/obstacle
R_CLEAN_GOOD = +10 # cleaning a dirty cell
R_CLEAN_BAD = -2   # cleaning a non-dirty cell
R_FINISH = +50     # bonus when all initial dirty cells are clean


Obstacles & Dirty Cells (From the Figure)

In [ ]:
# Part (a): Coordinates are (row, col), top-left is (0,0)

# Obstacles (red signs)
obstacles = {
    (1,2), (1,3), (1,7),
    (2,0),
    (3,3), (3,5),
    (4,1), (4,9),
    (5,1), (5,4),
    (6,1), (6,7),
    (7,7),
    (8,3), (8,4), (8,9),
    (9,6),
}

# Initially dirty cells (brooms)
dirty_init = {
    (0,2), (0,5), (0,9),
    (1,9),
    (2,3),
    (3,0), (3,7),
    (4,0),
    (5,2), (5,6),
    (6,6),
    (7,0), (7,2), (7,5),
    (8,8),
    (9,1),
}

print("Num obstacles:", len(obstacles))
print("Num dirty init:", len(dirty_init))


Z1:
 [[ 0.68]
 [-0.48]
 [ 0.18]]
A1:
 [[0.68]
 [0.  ]
 [0.18]]
Z2: -0.010000000000000037
y_hat: 0.49750002083312495
RMSE: 0.202499979166875


Environment State Initialization

In [ ]:
# Part (a): Environment state variables (initialized for an episode)

# Dirty status only for initially dirty cells
dirty_now = np.zeros((H, W), dtype=bool)
for (r,c) in dirty_init:
    dirty_now[r, c] = True

# Track last time a cell was cleaned (only meaningful for initial dirty cells)
last_clean_t = -np.ones((H, W), dtype=int)  # -1 means "not cleaned recently"

# Agent position and time
r, c = start_r, start_c
t = 0

dL/db2: -0.2499937501041652
dL/dW2:
 [[-0.16999575]
 [-0.        ]
 [-0.04499888]]


Simple Grid Print

In [ ]:
# Part (a): Quick visualization in text (R=robot, X=obstacle, D=dirty, .=empty)

grid = [["." for _ in range(W)] for _ in range(H)]
for (rr,cc) in obstacles:
    grid[rr][cc] = "X"
for (rr,cc) in dirty_init:
    grid[rr][cc] = "D"
grid[r][c] = "R"

for row in grid:
    print(" ".join(row))

### Part (b) - Q-Learning Update
Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]


Q-Learning Loop

In [ ]:
# Part (b): Q-Learning training (simple implementation)

alpha = 0.15
episodes = 2000
epsilon = 0.3
epsilon_min = 0.05
epsilon_decay = 0.999

# State: (r, c, dirty_here) => dirty_here in {0,1}
Q = np.zeros((H, W, 2, A), dtype=float)

q_returns = []
q_steps = []

for ep in range(episodes):
    # reset episode
    dirty_now = np.zeros((H, W), dtype=bool)
    for (rr,cc) in dirty_init:
        dirty_now[rr, cc] = True
    last_clean_t = -np.ones((H, W), dtype=int)

    r, c = start_r, start_c
    t = 0
    done = False
    total_reward = 0
    steps = 0

    while (not done) and steps < 5000:
        # current state
        dirty_here = 1 if dirty_now[r, c] else 0

        # epsilon-greedy action selection
        if np.random.rand() < epsilon:
            a = np.random.randint(A)
        else:
            a = int(np.argmax(Q[r, c, dirty_here, :]))

        # take action
        reward = R_STEP
        r2, c2 = r, c

        if a == 0:   # Up
            r2 = r - 1
        elif a == 1: # Down
            r2 = r + 1
        elif a == 2: # Left
            c2 = c - 1
        elif a == 3: # Right
            c2 = c + 1
        elif a == 4: # Clean
            if dirty_now[r, c]:
                dirty_now[r, c] = False
                last_clean_t[r, c] = t
                reward = R_CLEAN_GOOD
            else:
                reward = R_CLEAN_BAD

        # handle moves (invalid => stay)
        if a in [0,1,2,3]:
            if (r2 < 0) or (r2 >= H) or (c2 < 0) or (c2 >= W) or ((r2, c2) in obstacles):
                reward = R_INVALID
                r2, c2 = r, c
            else:
                r, c = r2, c2

        # time update
        t += 1
        steps += 1
        total_reward += reward

        # redirty dynamics
        for (rr,cc) in dirty_init:
            if last_clean_t[rr, cc] != -1 and (t - last_clean_t[rr, cc]) >= REDIRTY_STEPS:
                dirty_now[rr, cc] = True
                last_clean_t[rr, cc] = -1

        # check finish condition (all initial dirty cells are currently clean)
        all_clean = True
        for (rr,cc) in dirty_init:
            if dirty_now[rr, cc]:
                all_clean = False
                break
        if all_clean:
            total_reward += R_FINISH
            done = True

        # next state for update
        dirty_here2 = 1 if dirty_now[r, c] else 0
        best_next = np.max(Q[r, c, dirty_here2, :])

        # Q-learning update (note: state before action was (r2? actually we already moved r,c))
        # To keep it correct, we need the previous state indices. So we store them before acting:
        # We'll do it by using variables sr, sc, sd before action.

    # NOTE: We'll redo this loop in the next cell with a correct "previous state" update.
    q_returns.append(total_reward)
    q_steps.append(steps)

print("This cell sets the structure. Next cell provides the corrected Q-learning update with stored previous state.")


Q-Learning (Correct Update

In [ ]:
# Part (b): Q-Learning training (correct update with stored previous state)

alpha = 0.15
episodes = 2000
epsilon = 0.3
epsilon_min = 0.05
epsilon_decay = 0.999

Q = np.zeros((H, W, 2, A), dtype=float)

q_returns = []
q_steps = []

for ep in range(episodes):
    # reset episode
    dirty_now = np.zeros((H, W), dtype=bool)
    for (rr,cc) in dirty_init:
        dirty_now[rr, cc] = True
    last_clean_t = -np.ones((H, W), dtype=int)

    r, c = start_r, start_c
    t = 0
    done = False
    total_reward = 0
    steps = 0

    while (not done) and steps < 5000:
        # store previous state (important for correct update)
        sr, sc = r, c
        sd = 1 if dirty_now[sr, sc] else 0

        # epsilon-greedy
        if np.random.rand() < epsilon:
            a = np.random.randint(A)
        else:
            a = int(np.argmax(Q[sr, sc, sd, :]))

        # act
        reward = R_STEP
        r2, c2 = r, c

        if a == 0:   r2 = r - 1
        elif a == 1: r2 = r + 1
        elif a == 2: c2 = c - 1
        elif a == 3: c2 = c + 1
        elif a == 4:
            if dirty_now[r, c]:
                dirty_now[r, c] = False
                last_clean_t[r, c] = t
                reward = R_CLEAN_GOOD
            else:
                reward = R_CLEAN_BAD

        if a in [0,1,2,3]:
            if (r2 < 0) or (r2 >= H) or (c2 < 0) or (c2 >= W) or ((r2, c2) in obstacles):
                reward = R_INVALID
            else:
                r, c = r2, c2

        # time update
        t += 1
        steps += 1
        total_reward += reward

        # redirty
        for (rr,cc) in dirty_init:
            if last_clean_t[rr, cc] != -1 and (t - last_clean_t[rr, cc]) >= REDIRTY_STEPS:
                dirty_now[rr, cc] = True
                last_clean_t[rr, cc] = -1

        # finish?
        all_clean = True
        for (rr,cc) in dirty_init:
            if dirty_now[rr, cc]:
                all_clean = False
                break
        if all_clean:
            total_reward += R_FINISH
            done = True

        # next state
        sd2 = 1 if dirty_now[r, c] else 0

        # Q-learning update
        target = reward + gamma * np.max(Q[r, c, sd2, :])
        Q[sr, sc, sd, a] = Q[sr, sc, sd, a] + alpha * (target - Q[sr, sc, sd, a])

    q_returns.append(total_reward)
    q_steps.append(steps)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (ep+1) % 200 == 0:
        print(f"Q-Learning | ep {ep+1}/{episodes} | return={total_reward:.1f} | steps={steps} | eps={epsilon:.3f}")


### Part (c) - SARSA Update
Q(s,a) ← Q(s,a) + α [ r + γ Q(s',a') − Q(s,a) ]


SARSA Loop

In [ ]:
# Part (c): SARSA training (simple implementation)

alpha = 0.15
episodes = 2000
epsilon = 0.3
epsilon_min = 0.05
epsilon_decay = 0.999

Q_sarsa = np.zeros((H, W, 2, A), dtype=float)

sarsa_returns = []
sarsa_steps = []

for ep in range(episodes):
    # reset
    dirty_now = np.zeros((H, W), dtype=bool)
    for (rr,cc) in dirty_init:
        dirty_now[rr, cc] = True
    last_clean_t = -np.ones((H, W), dtype=int)

    r, c = start_r, start_c
    t = 0
    done = False
    total_reward = 0
    steps = 0

    # choose initial action (epsilon-greedy)
    sd = 1 if dirty_now[r, c] else 0
    if np.random.rand() < epsilon:
        a = np.random.randint(A)
    else:
        a = int(np.argmax(Q_sarsa[r, c, sd, :]))

    while (not done) and steps < 5000:
        # store previous state
        sr, sc = r, c
        sd = 1 if dirty_now[sr, sc] else 0

        # act using action a
        reward = R_STEP
        r2, c2 = r, c

        if a == 0:   r2 = r - 1
        elif a == 1: r2 = r + 1
        elif a == 2: c2 = c - 1
        elif a == 3: c2 = c + 1
        elif a == 4:
            if dirty_now[r, c]:
                dirty_now[r, c] = False
                last_clean_t[r, c] = t
                reward = R_CLEAN_GOOD
            else:
                reward = R_CLEAN_BAD

        if a in [0,1,2,3]:
            if (r2 < 0) or (r2 >= H) or (c2 < 0) or (c2 >= W) or ((r2, c2) in obstacles):
                reward = R_INVALID
            else:
                r, c = r2, c2

        # time update
        t += 1
        steps += 1
        total_reward += reward

        # redirty
        for (rr,cc) in dirty_init:
            if last_clean_t[rr, cc] != -1 and (t - last_clean_t[rr, cc]) >= REDIRTY_STEPS:
                dirty_now[rr, cc] = True
                last_clean_t[rr, cc] = -1

        # finish?
        all_clean = True
        for (rr,cc) in dirty_init:
            if dirty_now[rr, cc]:
                all_clean = False
                break
        if all_clean:
            total_reward += R_FINISH
            done = True

        # choose next action a' using epsilon-greedy (on-policy)
        sd2 = 1 if dirty_now[r, c] else 0
        if np.random.rand() < epsilon:
            a2 = np.random.randint(A)
        else:
            a2 = int(np.argmax(Q_sarsa[r, c, sd2, :]))

        # SARSA update uses Q(s', a')
        target = reward + gamma * Q_sarsa[r, c, sd2, a2]
        Q_sarsa[sr, sc, sd, a] = Q_sarsa[sr, sc, sd, a] + alpha * (target - Q_sarsa[sr, sc, sd, a])

        # move to next
        a = a2

    sarsa_returns.append(total_reward)
    sarsa_steps.append(steps)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (ep+1) % 200 == 0:
        print(f"SARSA      | ep {ep+1}/{episodes} | return={total_reward:.1f} | steps={steps} | eps={epsilon:.3f}")


Moving Average + Plot

In [ ]:
# Part (d): Plot average cumulative reward (moving average)

w = 50  # window for moving average

q_ma = np.convolve(np.array(q_returns), np.ones(w)/w, mode="valid")
s_ma = np.convolve(np.array(sarsa_returns), np.ones(w)/w, mode="valid")

plt.figure(figsize=(8,5))
plt.plot(q_ma, label="Q-Learning (avg return)")
plt.plot(s_ma, label="SARSA (avg return)")
plt.xlabel("Episode")
plt.ylabel("Average cumulative reward (window=50)")
plt.title("Q-Learning vs SARSA")
plt.grid(True)
plt.legend()
plt.show()

print("Q-Learning: final steps =", q_steps[-1], "| best (min) steps =", min(q_steps))
print("SARSA     : final steps =", sarsa_steps[-1], "| best (min) steps =", min(sarsa_steps))